# Sentinel-8D — Statistical Root-Cause Traceback

**Dataset:** CiP-DMD (TU Darmstadt) | **Version/date:** 2023 (DOI: 10.5281/zenodo.8420132) | **Source URL:** https://cloud.ptw-darmstadt.de/public.php/webdav

> Reruns top-to-bottom from raw data. Every number that appears in `reports/8D_Report.md` is produced by a cell here (execution.md §11).

Sections mirror the six analysis steps in `execution.md`. Fill each as you reach that day's guide.

## Setup

In [1]:
import sys
sys.path.append("..")  # make src/ importable from notebooks/

import pandas as pd
import numpy as np

from src import load, clean, stats

# Dataset selected via Day 1 Data Access Gate
DATASET = "cip_dmd"

## Step 0–1 — Data acquisition & understanding (Day 1–2)

See guides/day-1.md and guides/day-2.md. Load raw, build the data dictionary, reconstruct the routing, identify join keys.

In [ ]:
# Load raw data tables (quality CSVs + metadata JSONs)
raw = load.load_raw(DATASET)

# Build the data dictionary
data_dict = load.map_schema(raw, DATASET)
data_dict

In [ ]:
# Display the process flow diagram
from IPython.display import Image, display
display(Image(filename='../reports/figures/process_flow.png', width=900))

## Step 2 — Cleaning & tidying to one row per part (Day 2)

See guides/day-2.md. Output: `data/processed/parts.parquet`.

In [ ]:
# Tidy to one row per assembled cylinder
parts = clean.tidy_one_row_per_part(raw, data_dict)

# Handle missing values (report, add indicators, median-fill)
parts = clean.handle_missing(parts)

# Define binary failure label
parts = clean.define_label(parts)

# Save processed table
path = clean.save_processed(parts)
print(f'Saved {parts.shape} to {path}')
parts.head()

## Step 3 — Defect characterization / D2 evidence (Day 3)

See guides/day-3.md. Baseline rate + volume, Pareto of failure modes, pick the dominant mode. Save `reports/figures/pareto.png`.

In [ ]:
# --- Step 3: Baseline Defect Characterization & Pareto Analysis ---
total_parts = len(parts)
fail_count = int(parts["fail"].sum())
pass_count = total_parts - fail_count
defect_rate = (fail_count / total_parts) * 100
dpmo = (fail_count / total_parts) * 1_000_000

print("=== Baseline Metrics (8D Discipline D2) ===")
print(f"  Total Assembled Units (N):    {total_parts}")
print(f"  Conforming Parts (Pass):      {pass_count} ({pass_count/total_parts*100:.2f}%)")
print(f"  Defective Parts (Fail/Rework): {fail_count} ({defect_rate:.2f}%)")
print(f"  Defects Per Million (DPMO):   {dpmo:,.0f}")

# Generate and display Pareto chart
import subprocess
subprocess.run([sys.executable, "../scripts/gen_pareto.py"], check=True)
from IPython.display import Image, display
display(Image(filename="../reports/figures/pareto.png", width=850))

## Step 4 — Univariate screening (Day 3–4)

See guides/day-3.md / day-4.md. FDR + Bonferroni. Save `reports/figures/univariate_ranking.png`.

In [ ]:
# --- Step 4: Univariate Statistical Screening ---
# Test every upstream process & quality parameter for pass-vs-fail separation
ranking = stats.univariate_screen(parts, target="fail", alpha=0.05)

print(f"Screened {len(ranking)} parameters across Saw, Milling, and Lathe operations.")
print("")
print("=== Top Ranked Statistical Drivers (Sorted by Effect Size) ===")
display(ranking[["parameter", "type", "test", "statistic", "mean_diff", "effect_size_type", "effect_size", "p_fdr_bh", "significant_fdr"]])

# Shortlist parameters with significant FDR correction (p < 0.05) and non-trivial effect size (|d| >= 0.15 or Cramer V >= 0.1)
shortlist = ranking.loc[
    (ranking["significant_fdr"]) & (ranking["abs_effect_size"] >= 0.15),
    "parameter"
].tolist()
print(f"Shortlisted Candidates for Multivariate Isolation (Step 5): {shortlist}")

## Step 5 — Multivariate isolation + tree cross-check / D4 evidence (Day 4–5)

See guides/day-4.md / day-5.md. VIF, logistic odds ratios + CIs, tree importances; require model agreement.

In [ ]:
# TODO(day-4/5)
# stats.compute_vif(parts, shortlist)
# logit = stats.fit_logistic(parts, shortlist)
# importances = stats.tree_crosscheck(parts, shortlist)

## Step 6 — Root-cause confirmation & escape point (Day 5)

See guides/day-5.md. State root cause (station + parameter + condition, OR/CI/p). Physics sanity check. Escape point. Quantify the prize.

In [ ]:
# TODO(day-5): root-cause statement + the 'smoking-gun' figure

## Step 7 — Corrective-action design (Day 6)

See guides/day-6.md. SPC control at the offending operation: chart type, control limits, reaction plan.

In [ ]:
# TODO(day-6): SPC chart spec + control limits from the in-control subset